In [3]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load processed dataset from GitHub
url = "https://raw.githubusercontent.com/samarapires-ml/Buy-Now-Pay-Later--BNPL--Risk-Prediction/main/data/processed/clean_full_data.csv"
df = pd.read_csv(url)

# Preview data
df.head()

,Customer_Age,Annual_Income,Credit_Score,Purchase_Amount,Checkout_Time_Seconds,Gender_Male,Gender_Non-Binary,Purchase_Category_Electronics,Purchase_Category_Fashion,Purchase_Category_Groceries,...,BNPL_Provider_Klarna,BNPL_Provider_Sezzle,Device_Type_Mobile,Device_Type_Tablet,Connection_Type_VPN,Connection_Type_WiFi,Browser_Edge,Browser_Firefox,Browser_Safari,target
0,1.110297,-1.303034,-1.388519,-0.445029,-0.199028,True,False,False,False,False,...,False,True,False,True,False,True,False,True,False,0
1,0.371122,0.096571,-1.382224,-0.531422,-0.632426,True,False,False,False,True,...,False,False,True,False,False,True,False,True,False,0
2,-0.663723,0.422711,0.355240,1.482538,-0.120229,True,False,False,False,False,...,False,True,False,False,False,True,False,False,False,0
3,1.405967,0.778516,-0.651985,-0.627730,1.514864,True,False,False,True,False,...,False,True,True,False,False,False,False,False,False,0
4,-1.181145,-1.311090,-0.450540,1.821031,-1.065824,True,False,False,False,False,...,True,False,True,False,False,False,False,False,False,1


In [4]:
# Check all columns
df.columns

Index(['Customer_Age', 'Annual_Income', 'Credit_Score', 'Purchase_Amount',
       'Checkout_Time_Seconds', 'Gender_Male', 'Gender_Non-Binary',
       'Purchase_Category_Electronics', 'Purchase_Category_Fashion',
       'Purchase_Category_Groceries', 'Purchase_Category_Home & Furniture',
       'Purchase_Category_Travel', 'BNPL_Provider_Afterpay',
       'BNPL_Provider_Klarna', 'BNPL_Provider_Sezzle', 'Device_Type_Mobile',
       'Device_Type_Tablet', 'Connection_Type_VPN', 'Connection_Type_WiFi',
       'Browser_Edge', 'Browser_Firefox', 'Browser_Safari', 'target'],
      dtype='object')

In [5]:
# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Check shapes
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50000, 22)
y shape: (50000,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (40000, 22)
Test shape: (10000, 22)


In [7]:
# Select small feature set
selected_features = X_train.columns[:5]

X_train_small = X_train[selected_features]
X_test_small = X_test[selected_features]

# 🔥 SUBSET DATA (THIS IS THE MOST IMPORTANT STEP)
X_train_small = X_train_small.iloc[:1000]
y_train_small = y_train.iloc[:1000]

print("Subset shape:", X_train_small.shape)

Subset shape: (1000, 5)


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_small)
X_test_scaled = scaler.transform(X_test_small)

y_train_array = y_train_small.values
y_test_array = y_test.values

In [9]:
import pymc as pm

with pm.Model() as model:
    
    intercept = pm.Normal("intercept", mu=0, sigma=5)
    
    coefficients = pm.Normal(
        "coefficients", 
        mu=0, 
        sigma=2, 
        shape=X_train_scaled.shape[1]
    )
    
    logits = intercept + pm.math.dot(X_train_scaled, coefficients)
    
    likelihood = pm.Bernoulli(
        "likelihood",
        logit_p=logits,
        observed=y_train_array
    )
    
    trace = pm.sample(
        draws=150,
        tune=150,
        target_accept=0.9,
        cores=1,
        random_seed=42
    )

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, coefficients]


Output()

Sampling 2 chains for 150 tune and 150 draw iterations (300 + 300 draws total) took 1790 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


In [10]:
az.summary(trace)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
intercept,-1.270,0.087,-1.441,-1.121,0.004,0.004,464.0,162.0,1.00
coefficients[0],-0.082,0.076,-0.211,0.066,0.004,0.004,466.0,226.0,1.00
coefficients[1],-0.082,0.074,-0.225,0.061,0.003,0.004,569.0,252.0,1.00
coefficients[2],-0.668,0.075,-0.811,-0.533,0.004,0.004,465.0,186.0,1.00
coefficients[3],-0.039,0.082,-0.195,0.109,0.004,0.004,457.0,227.0,1.00
coefficients[4],0.051,0.076,-0.099,0.176,0.004,0.004,352.0,197.0,1.03


In [11]:
from sklearn.metrics import roc_auc_score

posterior = trace.posterior

intercept_samples = posterior["intercept"].values.flatten()
coef_samples = posterior["coefficients"].values.reshape(-1, X_train_scaled.shape[1])

intercept_mean = intercept_samples.mean()
coef_mean = coef_samples.mean(axis=0)

logits_test = intercept_mean + np.dot(X_test_scaled, coef_mean)
probs = 1 / (1 + np.exp(-logits_test))

auc = roc_auc_score(y_test_array, probs)
print("Bayesian Model AUC:", auc)

Bayesian Model AUC: 0.6798963098221316
